# 1. Метапрограммирование

**Метапрограммирование** - это написание программ, которые работают с другими программами или с самим кодом как с данными

Основные инструменты метапрограммирования в Python

**Декораторы**

Позволяют менять поведение функций или классов без изменения их исходного кода.

Применяются для логирования, проверки прав или замера времени выполнения. 

**Метаклассы**

В Python класс сам является объектом, экземпляром которого выступает метакласс (по умолчанию это type).

Позволяют вмешиваться в процесс создания классов, автоматически регистрировать их или менять структуру.

**Динамическое создание кода**

Функция type() может создавать новые классы «на лету», принимая имя, список базовых классов и словарь атрибутов.


# 2. Программирование над классами

## Динамический доступ к атрибутам объектов

In [1]:
class User:
    def __init__(self):
        self.name = "Ivan"
        self.age = 30

In [2]:
user = User()

Запросим атрибут

In [3]:
print(user.name)

Ivan


### **getattr()** позволяет получить атрибут программно

In [4]:
print(getattr(user, "name"))

Ivan


### **setattr()** позволяет создавать или изменять атрибуты программно

In [5]:
setattr(user, "status", "employed")

In [6]:
print(user.status)

employed


### **hasattr()** позволяет проверить наличие атрибута программно

In [7]:
hasattr(user, "status")

True

In [8]:
if hasattr(user, "email"):
    print(user.email)
else:
    print("Email отсутствует")

Email отсутствует


### **delattr()** позволяет удалить атрибут программно

In [9]:
delattr(user, "age")

In [10]:
hasattr(user, "age")

False

## Интроспекция

### **dir()** - посмотреть структуру объекта

In [11]:
print(dir(user))

['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__firstlineno__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__static_attributes__', '__str__', '__subclasshook__', '__weakref__', 'name', 'status']


### **\_\_dict\_\_** -  содержит внутренние атрибуты объекта

In [12]:
user.__dict__

{'name': 'Ivan', 'status': 'employed'}

In [13]:
user.__dict__["email"] = "ivan@example.com"

In [14]:
hasattr(user, "email")

True

In [15]:
print(user.email)

ivan@example.com


### **vars()** - альтернатива \_\_dict\_\_

In [16]:
vars(user)

{'name': 'Ivan', 'status': 'employed', 'email': 'ivan@example.com'}

## Динамическое создание классов

### type()

type() для определения типа:

In [17]:
type(user)

__main__.User

type() может динамически создавать классы:

In [18]:
User = type(
    "User",
    (),
    {
        "name": "Ivan"
    }
)

In [19]:
type(User)

type

In [20]:
user = User()
print(user.name)

Ivan


### Как работает type()

Форма:

```Python

type(
    class_name,
    base_classes,
    attributes
)

```

In [21]:
Person = type(
    "Person",
    (),
    {
        "name": "Ivan",
        "age": 30
    }
)

Здесь:

"Person" — имя класса

() — базовые классы

{ "name": "Ivan", "age": 30 } — атрибуты класса.

In [22]:
Person.__dict__

mappingproxy({'name': 'Ivan',
              'age': 30,
              '__module__': '__main__',
              '__dict__': <attribute '__dict__' of 'Person' objects>,
              '__weakref__': <attribute '__weakref__' of 'Person' objects>,
              '__doc__': None})

## Пример использования динамического создания классов

In [23]:
columns = {
    "age": [20, 25, 30, 35],
    "income": [50000, 60000, 75000, 90000],
    "score": [0.7, 0.8, 0.6, 0.9]
}

In [24]:
def create_stats_method(column_name):
    def stats(self):
        values = self.data[column_name]
        return {
            "column": column_name,
            "min": min(values),
            "max": max(values),
            "mean": sum(values) / len(values)
        }
    return stats

Функция create_stats_method() возвращает другую функцию.

Например, age_method = create_stats_method("age")

age_method — это функция, которая знает, что ей нужно работать с колонкой "age"



In [25]:
class_attributes = {
    "data": columns
}

In [26]:
for column_name in columns:
    method_name = f"{column_name}_stats"
    class_attributes[method_name] = create_stats_method(column_name)

В цикле получаем:

age     → age_stats

income  → income_stats

score   → score_stats

In [27]:
class_attributes

{'data': {'age': [20, 25, 30, 35],
  'income': [50000, 60000, 75000, 90000],
  'score': [0.7, 0.8, 0.6, 0.9]},
 'age_stats': <function __main__.create_stats_method.<locals>.stats(self)>,
 'income_stats': <function __main__.create_stats_method.<locals>.stats(self)>,
 'score_stats': <function __main__.create_stats_method.<locals>.stats(self)>}

In [28]:
Dataset = type(
    "Dataset",
    (),
    class_attributes
)

In [29]:
dataset = Dataset()

print(dataset.age_stats())
print(dataset.income_stats())
print(dataset.score_stats())

{'column': 'age', 'min': 20, 'max': 35, 'mean': 27.5}
{'column': 'income', 'min': 50000, 'max': 90000, 'mean': 68750.0}
{'column': 'score', 'min': 0.6, 'max': 0.9, 'mean': 0.75}


In [30]:
dataset.data

{'age': [20, 25, 30, 35],
 'income': [50000, 60000, 75000, 90000],
 'score': [0.7, 0.8, 0.6, 0.9]}

Python создал класс, приблизительно эквивалентный такому

```Python
class Dataset:

    def age_stats(self):
        ...

    def income_stats(self):
        ...

    def score_stats(self):
        ...
```

In [31]:
print(Dataset.__dict__)

{'data': {'age': [20, 25, 30, 35], 'income': [50000, 60000, 75000, 90000], 'score': [0.7, 0.8, 0.6, 0.9]}, 'age_stats': <function create_stats_method.<locals>.stats at 0x0000023C739DA3E0>, 'income_stats': <function create_stats_method.<locals>.stats at 0x0000023C739D9620>, 'score_stats': <function create_stats_method.<locals>.stats at 0x0000023C739DA160>, '__module__': '__main__', '__dict__': <attribute '__dict__' of 'Dataset' objects>, '__weakref__': <attribute '__weakref__' of 'Dataset' objects>, '__doc__': None}


## Пример: динамическое создание признаков (feature engineering)

In [32]:
import pandas as pd

In [35]:
class FeatureGenerator:
    def __init__(self, df):
        self.df = df.copy()
    
    def add_interaction_features(self, col1, col2):
        """Создаёт признак взаимодействия двух столбцов"""
        new_col_name = f"{col1}_x_{col2}"
        self.df[new_col_name] = self.df[col1] * self.df[col2]
        return self
    
    def add_polynomial_features(self, col, degree=2):
        """Создаёт полиномиальные признаки"""
        for d in range(2, degree + 1):
            new_col = f"{col}_pow_{d}"
            self.df[new_col] = self.df[col] ** d
        return self
    
    def get_df(self):
        return self.df

In [36]:
df = pd.DataFrame({
    "age": [20, 25, 30, 35],
    "income": [50000, 60000, 75000, 90000],
    "score": [0.7, 0.8, 0.6, 0.9]
})

In [37]:
generator = FeatureGenerator(df)

In [38]:
df_new = (
    generator
    .add_interaction_features('age', 'income')
    .add_polynomial_features('score', degree=3)
    .get_df()
)

In [39]:
df_new

,age,income,score,age_x_income,score_pow_2,score_pow_3
0,20,50000,0.7,1000000,0.49,0.343
1,25,60000,0.8,1500000,0.64,0.512
2,30,75000,0.6,2250000,0.36,0.216
3,35,90000,0.9,3150000,0.81,0.729


# 3. Метаклассы

## Метакласс - это класс, который создает классы

In [40]:
# Класс User создал объект user
user = User()

In [41]:
# Какой метакласс у обычного класса?
print(type(User))

<class 'type'>


type — стандартный метакласс Python

In [66]:
# Метакласс обычно наследуется от type

In [42]:
class MyMeta(type):
    def __new__(cls, name, bases, namespace):
        print(f"Создается класс {name}")
        
        return super().__new__(
            cls,
            name,
            bases,
            namespace
        )

In [43]:
class User(metaclass=MyMeta):
    pass

Создается класс User


In [44]:
print(type(User))

<class '__main__.MyMeta'>


Метакласс получает контроль над созданием класса

## \_\_new\_\_

\_\_new\_\_ - это специальный метод Python, который отвечает за создание нового объекта

\_\_new\_\_() создает объект 

\_\_init\_\_() настраивает объект

cls — это класс, объект которого сейчас создается

self — это уже созданный экземпляр

In [45]:
# Здесь new ничего не создает
class Person:
    def __new__(cls):
        print("Создание")

In [46]:
person = Person()

Создание


In [47]:
type(person)

NoneType

In [48]:
# new должен вернуть объект
class Person:
    def __new__(cls):
        return super().__new__(cls)

In [49]:
person = Person()
type(person)

__main__.Person

In [50]:
class MyMeta(type):
    def __new__(cls, name, bases, namespace):
        print(f"Создается класс {name}")
        
        return super().__new__(
            cls,
            name,
            bases,
            namespace
        )

In [51]:
class Person(metaclass=MyMeta):
    age = 30

Создается класс Person


In [52]:
person = Person()

In [53]:
type(person)

__main__.Person

## Пример использования метакласса

In [54]:
class ModelMeta(type):
    """Метакласс для автоматической регистрации моделей."""

    registry = {}

    def __new__(cls, name, bases, namespace):
        print(f"Создаем класс: {name}")

        # Сначала создаем сам класс
        new_class = super().__new__(
            cls,
            name,
            bases,
            namespace
        )

        # Регистрируем созданный класс
        cls.registry[name] = new_class

        return new_class

In [55]:
class BaseModel(metaclass=ModelMeta):
    pass

Создаем класс: BaseModel


In [56]:
ModelMeta.registry

{'BaseModel': __main__.BaseModel}

In [57]:
class LinearRegression(BaseModel):
    pass

Создаем класс: LinearRegression


In [58]:
ModelMeta.registry

{'BaseModel': __main__.BaseModel,
 'LinearRegression': __main__.LinearRegression}

In [59]:
class RandomForest(BaseModel):
    pass

Создаем класс: RandomForest


In [60]:
class NeuralNetwork(BaseModel):
    pass

Создаем класс: NeuralNetwork


In [61]:
print("Зарегистрированные модели:")
for name in ModelMeta.registry:
    print(name)

Зарегистрированные модели:
BaseModel
LinearRegression
RandomForest
NeuralNetwork


## \_\_init_subclass\_\_

In [62]:
class BaseModel:
    def __init_subclass__(cls):
        print(f"Создан подкласс {cls.__name__}")

In [63]:
class CatModel(BaseModel):
    pass

Создан подкласс CatModel


**Регистрация моделей с subclass**

In [64]:
class Model:
    registry = {}
    def __init_subclass__(cls, **kwargs):
        super().__init_subclass__(**kwargs)
        Model.registry[cls.__name__] = cls

In [65]:
class RandomForestModel(Model):
    pass

class XGBoostModel(Model):
    pass

class NeuralNetworkModel(Model):
    pass

In [66]:
print(Model.registry)

{'RandomForestModel': <class '__main__.RandomForestModel'>, 'XGBoostModel': <class '__main__.XGBoostModel'>, 'NeuralNetworkModel': <class '__main__.NeuralNetworkModel'>}


Если задача заключается просто в том, чтобы:

- регистрировать классы;
- проверять наследников;
- автоматически добавлять некоторую логику;
- контролировать создание подклассов

часто достаточно \_\_init_subclass\_\_

Метакласс нужен для более глубокого контроля

Можно запомнить:
- декоратор изменяет функцию/класс
- \_\_init_subclass\_\_ контролирует создание подклассов
- метакласс контролирует создание классов

# 4. dataclasses

In [67]:
class User:
    def __init__(self, name, age):
        self.name = name
        self.age = age

In [68]:
from dataclasses import dataclass

In [69]:
@dataclass
class User:
    name: str
    age: int

Python автоматически создает:

- \_\_init\_\_()
- \_\_repr\_\_()
- \_\_eq\_\_()

и другие методы в зависимости от параметров.

То есть достаточно описать структуру данных, а Python автоматически сгенерирует часть кода

Пример: создаем конфигурацию модели

In [70]:
@dataclass
class TrainingConfig:
    learning_rate: float
    batch_size: int
    epochs: int
    model_name: str

In [71]:
# Создаем конкретный экземпляр
config = TrainingConfig(
    learning_rate=0.001,
    batch_size=64,
    epochs=20,
    model_name="transformer"
)

In [72]:
config

TrainingConfig(learning_rate=0.001, batch_size=64, epochs=20, model_name='transformer')